## **Step 1: Mount Google Drive**

In [1]:
# from google.colab import drive

# drive.mount('/content/drive')

## **Step 2: Set Dataset Path**

In [2]:
# dataset_path = "/content/drive/MyDrive/Uni Life/Junior/3rd Term/Deep Learning/Project/Dataset"

dataset_path = "/Users/leonmarco/Downloads/Dataset"

## **Step 3: Install & Import Libraries**

In [3]:
%pip install ultralytics matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, shutil, random
from pathlib import Path

## **Step 4: Prepare Dataset for YOLOv8**

YOLOv8 classification expects the following folder structure:

```
dataset/
├── train/
│   ├── Improperly Wearing Facemask/
│   ├── Not Wearing Facemask/
│   └── Wearing Facemask/
└── val/
    ├── Improperly Wearing Facemask/
    ├── Not Wearing Facemask/
    └── Wearing Facemask/
```

The cell below copies the original dataset into this structure with an 80/20 split.

In [5]:
split_dataset_path = os.path.join(os.path.dirname(dataset_path), "Dataset_YOLOv8")

# Only run the split if it hasn't been done yet
if not os.path.exists(split_dataset_path):
    random.seed(42)
    val_ratio = 0.2

    for class_name in os.listdir(dataset_path):
        class_dir = os.path.join(dataset_path, class_name)
        if not os.path.isdir(class_dir) or class_name.startswith("."):
            continue

        images = [f for f in os.listdir(class_dir)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
        random.shuffle(images)

        split_idx = int(len(images) * (1 - val_ratio))
        train_imgs = images[:split_idx]
        val_imgs = images[split_idx:]

        for subset, img_list in [("train", train_imgs), ("val", val_imgs)]:
            dest = os.path.join(split_dataset_path, subset, class_name)
            os.makedirs(dest, exist_ok=True)
            for img in img_list:
                shutil.copy2(os.path.join(class_dir, img), dest)

    print(f"Dataset split created at: {split_dataset_path}")
else:
    print(f"Dataset split already exists at: {split_dataset_path}")

# Print summary
for subset in ["train", "val"]:
    subset_dir = os.path.join(split_dataset_path, subset)
    if os.path.exists(subset_dir):
        for cls in sorted(os.listdir(subset_dir)):
            cls_path = os.path.join(subset_dir, cls)
            if os.path.isdir(cls_path):
                count = len([f for f in os.listdir(cls_path)
                             if not f.startswith(".")])
                print(f"  {subset}/{cls}: {count} images")

Dataset split already exists at: /Users/leonmarco/Downloads/Dataset_YOLOv8
  train/Improperly Wearing Facemask: 144 images
  train/Not Wearing Facemask: 187 images
  train/Wearing Facemask: 160 images
  val/Improperly Wearing Facemask: 36 images
  val/Not Wearing Facemask: 47 images
  val/Wearing Facemask: 40 images


## **Step 5: Load YOLOv8 Classification Model**

We use the **YOLOv8 Nano Classification** model (`yolov8n-cls.pt`) pretrained on ImageNet.
This gives us powerful feature extractors that we fine-tune on our small dataset.

| Model | Size | ImageNet Top-1 | Parameters |
|-------|------|----------------|------------|
| yolov8n-cls | 5.4 MB | 69.0% | 2.7M |
| yolov8s-cls | 11.4 MB | 73.8% | 6.4M |
| yolov8m-cls | 33.4 MB | 76.8% | 17.0M |

In [6]:
# Load pretrained YOLOv8 nano classification model
model = YOLO("yolov8n-cls.pt")

print(f"Model: {model.model_name}")
print(f"Task: {model.task}")

Model: yolov8n-cls.pt
Task: classify


## **Step 6: Train Model**

YOLOv8 handles everything automatically during training:
- **Data augmentation** (flip, rotation, color jitter, mixup, mosaic)
- **Learning rate scheduling** (warmup + cosine annealing)
- **Early stopping** via `patience` parameter
- **Best model checkpointing**

In [7]:
results = model.train(
    data=split_dataset_path,
    epochs=50,
    imgsz=128,
    batch=32,
    patience=10,         # Early stopping: stop if val doesn't improve for 10 epochs
    pretrained=True,     # Use ImageNet pretrained weights
    optimizer="Adam",
    lr0=0.001,           # Initial learning rate
    dropout=0.2,         # Dropout for regularization
    verbose=True,
    project="runs/classify",
    name="mask_detector",
    exist_ok=True,
)

Ultralytics 8.4.60 🚀 Python-3.11.13 torch-2.12.0 CPU (Apple M3)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/leonmarco/Downloads/Dataset_YOLOv8, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mask_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=

## **Step 7: Evaluate Model**

In [8]:
# Validate on the validation set
metrics = model.val()

print(f"Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")

Ultralytics 8.4.60 🚀 Python-3.11.13 torch-2.12.0 CPU (Apple M3)
YOLOv8n-cls summary (fused): 30 layers, 1,438,723 parameters, 0 gradients, 3.3 GFLOPs
train: /Users/leonmarco/Downloads/Dataset_YOLOv8/train... found 491 images in 3 classes ✅ 
val: /Users/leonmarco/Downloads/Dataset_YOLOv8/val... found 123 images in 3 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4387.8±2710.4 MB/s, size: 536.8 KB)
val: Scanning /Users/leonmarco/Downloads/Dataset_YOLOv8/val... 123 images, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 57.3Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 8/8 2.3it/s 3.4s0.5ss
                   all      0.992          1
Speed: 0.0ms preprocess, 4.7ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /Users/leonmarco/Programming/CCDEPLRL_COM232_PROJECT/runs/classify/val
Top-1 Accuracy: 0.9919
Top-5 Accuracy: 1.0000


## **Step 8: Plot Training Curves**

YOLOv8 saves training metrics to a CSV file. We read it and plot accuracy & loss curves.

In [9]:
# Find the results CSV
results_csv = Path("runs/classify/mask_detector/results.csv")

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # Remove whitespace from column names
    print("Available columns:", list(df.columns))
else:
    print(f"Results file not found at {results_csv}")
    print("Make sure training has completed successfully.")

Results file not found at runs/classify/mask_detector/results.csv
Make sure training has completed successfully.


In [10]:
# Plot Training & Validation Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(df["epoch"], df["train/loss"], label="Train Loss", linewidth=2)
axes[0].plot(df["epoch"], df["val/loss"], label="Validation Loss", linewidth=2)
axes[0].set_title("Training & Validation Loss", fontsize=14)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(df["epoch"], df["metrics/accuracy_top1"], label="Top-1 Accuracy", linewidth=2)
axes[1].plot(df["epoch"], df["metrics/accuracy_top5"], label="Top-5 Accuracy", linewidth=2)
axes[1].set_title("Training Accuracy", fontsize=14)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

## **Step 9: Confusion Matrix**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Get class names from the dataset directory
val_dir = os.path.join(split_dataset_path, "val")
class_names = sorted([d for d in os.listdir(val_dir)
                      if os.path.isdir(os.path.join(val_dir, d)) and not d.startswith(".")])

print("Class names:", class_names)

# Predict on all validation images
y_true = []
y_pred = []

for cls_idx, cls_name in enumerate(class_names):
    cls_dir = os.path.join(val_dir, cls_name)
    for img_file in os.listdir(cls_dir):
        if img_file.startswith("."):
            continue
        img_path = os.path.join(cls_dir, img_file)
        result = model.predict(img_path, verbose=False)
        pred_idx = result[0].probs.top1
        y_true.append(cls_idx)
        y_pred.append(pred_idx)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Build and plot the confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# Print detailed classification report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

## **Step 10: Save the Model**

YOLOv8 automatically saves the best model during training at:
`runs/classify/mask_detector/weights/best.pt`

We copy it to the project root for use with `app.py`.

In [ ]:
import shutil

best_model_path = Path("runs/classify/mask_detector/weights/best.pt")
dest_path = Path("best.pt")

if best_model_path.exists():
    shutil.copy2(best_model_path, dest_path)
    print(f"Best model copied to: {dest_path.resolve()}")
    print(f"Model size: {dest_path.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print(f"Best model not found at {best_model_path}")
    print("Make sure training has completed successfully.")

# Print class name mapping
print("\nClass mapping:")
test_model = YOLO(str(dest_path))
print(test_model.names)

## **Test**

In [ ]:
# Quick test: predict on a single image
# Uncomment and set the path to an image you want to test

# test_img = "/path/to/test/image.jpg"
# result = model.predict(test_img, verbose=False)
# probs = result[0].probs
# 
# print(f"Predicted class: {model.names[probs.top1]}")
# print(f"Confidence: {probs.top1conf:.4f}")
# print(f"\nAll probabilities:")
# for idx, name in model.names.items():
#     print(f"  {name}: {probs.data[idx]:.4f}")